# Pixels to Predictions: DL Vision Challenge

**Model:** `HuggingFaceTB/SmolVLM-500M-Instruct`  
**Fine-tuning:** QLoRA (4-bit NF4, ≤5M trainable params)  
**Scoring:** Multiple-choice log-likelihood

---

## 📓 Experiment Diary — Run Start

> **Pre-run notes:**

In [ ]:
RUN_ID = "run_4_3"
SUB_RUN_ID = "using_run_5"  # can also be blank

In [ ]:
import datetime
import os, json
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
os.makedirs("diary", exist_ok=True)
with open("diary/compute_diary.txt", "a") as f:
    f.write(json.dumps({"event": "timestamp", "when": stamp}) + "\n")
print("⏱️ Compute diary timestamp:", stamp)

⏱️ Compute diary timestamp: 2026-05-06 20:35:54


## 0. Imports & Environment

In [ ]:
# ── 0a. Install dependencies (uncomment once per environment) ─────────────────
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 145.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import math
import time
import random
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType

print("All imports OK.")

All imports OK.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR_PATH = '/content/drive/MyDrive/Colab Notebooks/' + RUN_ID + '/'
DRIVE_DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/pixels-to-predictions.zip'

# Paths — adjust to Colab layout

# Local path for faster access
LOCAL_DATA_DIR = Path("/content/data")

# Do not recopy if data already exists
if not os.path.exists(LOCAL_DATA_DIR):
    # Ensure local data directory exists
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    # Copy data from Drive to local disk
    print(f"Copying data from {DRIVE_DATA_PATH} to disk...")
    !cp -r "{DRIVE_DATA_PATH}" "."
    !unzip -q "pixels-to-predictions.zip" -d "{LOCAL_DATA_DIR}"
    print("Data copy complete.")

# Use the local data directory for the rest of the notebook
DATA_DIR = LOCAL_DATA_DIR / Path("pixels-to-predictions/")
print(f'All data present at: {DATA_DIR}')

Mounted at /content/drive
Copying data from /content/drive/MyDrive/Colab Notebooks/pixels-to-predictions.zip to disk...
Data copy complete.
All data present at: /content/data/pixels-to-predictions


In [ ]:
# ── 0b-ii. Capture & save library versions for reproducibility ────────────────
import sys, platform, transformers, peft, PIL, accelerate
try:
    import bitsandbytes as bnb
    bnb_ver = bnb.__version__
except Exception:
    bnb_ver = "n/a"

env_info = {
    "recorded_at":    datetime.datetime.now().isoformat(),
    "python":         sys.version,
    "platform":       platform.platform(),
    "torch":          torch.__version__,
    "cuda":           torch.version.cuda if torch.cuda.is_available() else "n/a",
    "transformers":   transformers.__version__,
    "peft":           peft.__version__,
    "pillow":         PIL.__version__,
    "accelerate":     accelerate.__version__,
    "bitsandbytes":   bnb_ver,
    "numpy":          np.__version__,
    "pandas":         pd.__version__,
}

with open(DRIVE_DIR_PATH + "environment.json", "w") as f:
    json.dump(env_info, f, indent=2)

print("Library versions saved to environment.json")
for k, v in env_info.items():
    print(f"  {k:<16}: {v}")

Library versions saved to environment.json
  recorded_at     : 2026-05-06T20:37:19.766096
  python          : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
  platform        : Linux-6.6.113+-x86_64-with-glibc2.35
  torch           : 2.10.0+cu128
  cuda            : 12.8
  transformers    : 4.57.6
  peft            : 0.18.1
  pillow          : 11.3.0
  accelerate      : 1.13.0
  bitsandbytes    : 0.49.2
  numpy           : 2.0.2
  pandas          : 2.2.2


In [ ]:
# ── 0c. Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Makes CUDA ops deterministic (slight perf cost — fine for a class project)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Global seed set to {SEED}")

Global seed set to 42


In [ ]:
# ── 0d. Device & VRAM ─────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU    : {gpu_name}")
    print(f"VRAM   : {total_vram:.2f} GB")
else:
    print("No GPU found — running on CPU (expect slow training).")

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB
VRAM   : 39.49 GB


In [ ]:
# ── 0e. BitsAndBytes/QLoRA config ─────────────────────────────────────────────────────────
use_bnb = True # Flag to enable/disable BitsAndBytes quantization

# Determine the dtypes based on bf16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    # If bf16 is supported, use it where applicable
    amp_autocast_dtype = torch.bfloat16
    bnb_compute_dtype = torch.bfloat16
    model_load_dtype = torch.bfloat16
    print("CUDA device supports bfloat16. Using bfloat16 for AMP and QLoRA compute dtype.")
else:
    # If bf16 is NOT supported, use float32 for bnb_4bit_compute_dtype as requested.
    # For AMP autocast and model loading, stick with float16 if cuda is available,
    # otherwise float32 (which is already handled by existing conditionals for CPU).
    amp_autocast_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    bnb_compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_load_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print("CUDA device does NOT support bfloat16.")
    print(f"  Using {amp_autocast_dtype} for AMP autocast.")
    print(f"  Using {bnb_compute_dtype} for bnb_4bit_compute_dtype (as requested).")
    print(f"  Using {model_load_dtype} for model loading.")

CUDA device supports bfloat16. Using bfloat16 for AMP and QLoRA compute dtype.


## 1. Hyperparameters

In [ ]:
import torch

# ── 1. Define hyperparameters — edit values here, then run this cell ──────────

# Model
MODEL_ID            = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Data
IMG_SIZE            = 384
MAX_SEQ_LEN         = 1024
TRAIN_SUBSET_FRAC   = 1.0    # fraction of train set to use; 1.0 = full dataset

# Training
TRAIN_BATCH         = 4
GRAD_ACCUM          = 4      # effective batch = TRAIN_BATCH * GRAD_ACCUM
EVAL_BATCH          = 8
LR                  = 2e-4
EPOCHS              = 5
WARMUP_RATIO        = 0.05

# LoRA
LORA_RANK           = 8
LORA_ALPHA          = 16
LORA_DROPOUT        = 0.05
# LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# ── Snapshot to dict (used throughout for logging) ────────────────────────────
cfg = {
    "MODEL_ID":            MODEL_ID,
    "IMG_SIZE":            IMG_SIZE,
    "MAX_SEQ_LEN":         MAX_SEQ_LEN,
    "TRAIN_SUBSET_FRAC":   TRAIN_SUBSET_FRAC,
    "TRAIN_BATCH":         TRAIN_BATCH,
    "GRAD_ACCUM":          GRAD_ACCUM,
    "EVAL_BATCH":          EVAL_BATCH,
    "LR":                  LR,
    "EPOCHS":              EPOCHS,
    "WARMUP_RATIO":        WARMUP_RATIO,
    "LORA_RANK":           LORA_RANK,
    "LORA_ALPHA":          LORA_ALPHA,
    "LORA_DROPOUT":        LORA_DROPOUT,
    "LORA_TARGET_MODULES": LORA_TARGET_MODULES,
    "use_bnb":             use_bnb,
    "amp_autocast_dtype":  str(amp_autocast_dtype).replace("torch.", ""),
    "bnb_compute_dtype":   str(bnb_compute_dtype).replace("torch.", ""),
    "model_load_dtype":    str(model_load_dtype).replace("torch.", ""),
}

# ── Save to disk ──────────────────────────────────────────────────────────────
with open(DRIVE_DIR_PATH + f"config{SUB_RUN_ID}.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("config.json saved:")
print(json.dumps(cfg, indent=2))

config.json saved:
{
  "MODEL_ID": "HuggingFaceTB/SmolVLM-500M-Instruct",
  "IMG_SIZE": 384,
  "MAX_SEQ_LEN": 1024,
  "TRAIN_SUBSET_FRAC": 1.0,
  "TRAIN_BATCH": 4,
  "GRAD_ACCUM": 4,
  "EVAL_BATCH": 8,
  "LR": 0.0002,
  "EPOCHS": 5,
  "WARMUP_RATIO": 0.05,
  "LORA_RANK": 8,
  "LORA_ALPHA": 16,
  "LORA_DROPOUT": 0.05,
  "LORA_TARGET_MODULES": [
    "q_proj",
    "v_proj",
    "k_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
  ],
  "use_bnb": true,
  "amp_autocast_dtype": "bfloat16",
  "bnb_compute_dtype": "bfloat16",
  "model_load_dtype": "bfloat16"
}


## 2. Dataset

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# 'choices' is stored as a JSON string — parse it into a Python list
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train : {len(train_df):,} rows")
print(f"Val   : {len(val_df):,} rows")
print(f"Test  : {len(test_df):,} rows")

# ── Optional training subset ──────────────────────────────────────────────────
if TRAIN_SUBSET_FRAC < 1.0:
    train_df = train_df.sample(frac=TRAIN_SUBSET_FRAC, random_state=SEED).reset_index(drop=True)
    print(f"Using {TRAIN_SUBSET_FRAC:.0%} subset → {len(train_df):,} training rows")

    val_df = val_df.sample(frac=TRAIN_SUBSET_FRAC, random_state=SEED).reset_index(drop=True)
    print(f"Using {TRAIN_SUBSET_FRAC:.0%} subset → {len(val_df):,} validation rows")



Train : 3,109 rows
Val   : 1,048 rows
Test  : 1,008 rows


In [ ]:
# ── 2b. Prompt template ───────────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Build the text prompt for the VLM.
    The <image> token tells the model where to inject vision features.

    Parameters
    ----------
    row            : a single DataFrame row (pd.Series)
    include_answer : if True, append the ground-truth letter (for training)
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(row["choices"])
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt


# ── Sanity-check: print one prompt ───────────────────────────────────────────
print("=== SAMPLE TRAINING PROMPT ===")
print(build_prompt(train_df.iloc[0], include_answer=True))
print()
print("=== SAMPLE INFERENCE PROMPT ===")
print(build_prompt(val_df.iloc[0], include_answer=False))

=== SAMPLE TRAINING PROMPT ===
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete fo

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    """
    Loads images from disk and builds prompts on-the-fly.
    is_train=True  → returns full prompt with answer appended (for causal LM loss)
    is_train=False → returns prompt without answer + choice list (for MC scoring)
    """

    def __init__(
        self,
        df: pd.DataFrame,
        data_dir: Path,
        img_size: int = 224,
        is_train: bool = True,
    ):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.img_size  = img_size
        self.is_train  = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        full_path = self.data_dir / rel_path
        img = Image.open(full_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "id":      row["id"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }


train_ds = ScienceQADataset(train_df, DATA_DIR, IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, IMG_SIZE, is_train=False)

print(f"train_ds : {len(train_ds):,}")
print(f"val_ds   : {len(val_ds):,}")
print(f"test_ds  : {len(test_ds):,}")

train_ds : 3,109
val_ds   : 1,048
test_ds  : 1,008


## 3. Model + QLoRA

In [ ]:
# ── 3a. Processor ─────────────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.max_image_tiles = 1  # disable splitting; 1 tile = base patches only
processor.image_processor.do_image_splitting = False

# Ensure pad token is defined (required for batched inference)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Vocab size : {processor.tokenizer.vocab_size:,}")
print(f"Pad token  : '{processor.tokenizer.pad_token}'")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Vocab size : 49,152
Pad token  : '<|im_end|>'


In [ ]:
# ── 3b. 4-bit Quantization Config (QLoRA) ────────────────────────────────────
# NF4 quantization keeps the backbone frozen in 4-bit;
# only the LoRA adapter weights (float16) are trained.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal-Float 4 — best for LLMs
    bnb_4bit_use_double_quant=True,     # Nested quantization saves ~0.4 bit/param
    bnb_4bit_compute_dtype=bnb_compute_dtype,
)

print("BitsAndBytes config ready.")

BitsAndBytes config ready.


In [ ]:
# ── 3c. Load Pretrained HF Model ─────────────────────────────────────────────────────────────
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if use_bnb else None,
    dtype=model_load_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)

if not torch.cuda.is_available():
    model = model.to(device)

# Disable caching — incompatible with gradient checkpointing
model.config.use_cache = False

print("Base model loaded.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Base model loaded.


##### Image Token Diagnostic Functions START

In [ ]:
# DIAGNOSTICS: Check how many tokens are generated for a dummy image

def check_image_token_length():
    dummy = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
    enc   = processor(text=["<image>"], images=[dummy], return_tensors="pt")

    image_token_id   = processor.tokenizer.convert_tokens_to_ids("<image>")
    n_image_tokens   = int((enc["input_ids"] == image_token_id).sum().item())
    total_tokens     = int(enc["input_ids"].shape[1])

    print(f"image token id   : {image_token_id}")
    print(f"image tokens     : {n_image_tokens}")
    print(f"total tokens     : {total_tokens}")
    print(f"pixel_values     : {enc['pixel_values'].shape}")

In [ ]:
# DIAGNOSTICS: Read all config values in one place

def print_vision_diagnostics():
    PATCH_SIZE    = model.config.vision_config.patch_size          # fixed by arch (14)
    SHUFFLE       = model.model.connector.scale_factor             # baked in at load time
    proc_seq_len  = processor.image_seq_len                        # what encoder produces
    proc_max_size = processor.image_processor.max_image_size       # internal resize target
    do_splitting  = processor.image_processor.do_image_splitting   # tile splitting flag

    print("═" * 55)
    print("  Vision pipeline config")
    print("═" * 55)
    print(f"  IMG_SIZE (dataset resize)     : {IMG_SIZE}")
    print(f"  patch_size (arch, fixed)      : {PATCH_SIZE}")
    print(f"  connector.scale_factor        : {SHUFFLE}")
    print(f"  processor.image_seq_len       : {proc_seq_len}")
    print(f"  processor.max_image_size      : {proc_max_size}")
    print(f"  processor.size                : {processor.image_processor.size}")
    print(f"  processor.do_image_splitting  : {do_splitting}")
    print(f"  processor.max_image_tiles     : {processor.image_processor.max_image_tiles}")

##### Image Token Diagnostic Functions END

In [ ]:
check_image_token_length()

image token id   : 49190
image tokens     : 64
total tokens     : 67
pixel_values     : torch.Size([1, 1, 3, 512, 512])


In [ ]:
print_vision_diagnostics()

═══════════════════════════════════════════════════════
  Vision pipeline config
═══════════════════════════════════════════════════════
  IMG_SIZE (dataset resize)     : 384
  patch_size (arch, fixed)      : 16
  connector.scale_factor        : 4
  processor.image_seq_len       : 64
  processor.max_image_size      : {'longest_edge': 512}
  processor.size                : {'longest_edge': 2048}
  processor.do_image_splitting  : False
  processor.max_image_tiles     : 1


In [ ]:
# # ── 3d. Set Processor Image Size ─────────────────────────────────────────────────────────────

processor.image_processor.max_image_size = {"longest_edge": IMG_SIZE}
processor.image_processor.size           = {"longest_edge": IMG_SIZE}
processor.image_processor.do_image_splitting = False

# NOTE:
# processor.image_seq_len must match what the connector actually produces
# for the given IMG_SIZE: (patches_per_side / scale_factor)^2

PATCH_SIZE   = model.config.vision_config.patch_size  # 16
SCALE_FACTOR = model.config.scale_factor              # 4
patches_per_side          = IMG_SIZE // PATCH_SIZE    # ex: 128//16 = 8
processor.image_seq_len   = (patches_per_side // SCALE_FACTOR) ** 2  # ex: (8//4)^2 = 4

print(f"patches_per_side       : {patches_per_side}")
print(f"processor.image_seq_len: {processor.image_seq_len}")

patches_per_side       : 24
processor.image_seq_len: 36


In [ ]:
print_vision_diagnostics()

═══════════════════════════════════════════════════════
  Vision pipeline config
═══════════════════════════════════════════════════════
  IMG_SIZE (dataset resize)     : 384
  patch_size (arch, fixed)      : 16
  connector.scale_factor        : 4
  processor.image_seq_len       : 36
  processor.max_image_size      : {'longest_edge': 384}
  processor.size                : {'longest_edge': 384}
  processor.do_image_splitting  : False
  processor.max_image_tiles     : 1


In [ ]:
check_image_token_length()

image token id   : 49190
image tokens     : 36
total tokens     : 39
pixel_values     : torch.Size([1, 1, 3, 384, 384])


In [ ]:
# ── 3e. LoRA config + PEFT wrapping ──────────────────────────────────────────
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_cfg)

# ── Parameter count ───────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}  "
      f"({100 * trainable_params / total_params:.3f}%)")
assert trainable_params <= 5_000_000, (
    f"Trainable params {trainable_params:,} exceed the 5M competition limit!"
)

Total params     : 306,614,464
Trainable params : 4,784,128  (1.560%)


## 4. Collate Functions

In [ ]:
# ── 4a. Training collate ──────────────────────────────────────────────────────
# The full prompt (including the answer letter) is fed in.
# Labels = input_ids, but prompt tokens are masked to -100
# so the cross-entropy loss is computed ONLY on the answer token.

def train_collate(batch: list[dict]) -> dict:
    images = [item["image"] for item in batch]
    texts  = [item["text"]  for item in batch]

    # Tokenise the full prompt+answer text together with the image
    encoding = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )

    input_ids      = encoding["input_ids"]           # (B, L)
    attention_mask = encoding["attention_mask"]       # (B, L)

    # ── Build labels: mask everything except the answer token(s) ──────────────
    # Strategy: tokenise just the prompt (no answer), find its length,
    # then mask [0 .. prompt_len-1] → -100 in labels.
    prompt_texts = []
    for item in batch:
        # The prompt without the answer letter (ends with "Answer:")
        full_text   = item["text"]           # "... Answer: B"
        answer_idx  = item["answer"]
        answer_letter = CHOICE_LETTERS[answer_idx]
        # Strip the trailing " X" to recover the bare prompt
        prompt_only = full_text[: full_text.rfind(f" {answer_letter}")]
        prompt_texts.append(prompt_only)

    prompt_enc = processor(
        text=prompt_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    prompt_lens = prompt_enc["attention_mask"].sum(dim=1)  # (B,) actual token counts

    labels = input_ids.clone()
    for i, plen in enumerate(prompt_lens):
        labels[i, :plen] = -100          # mask prompt tokens
    # Also mask padding tokens
    labels[attention_mask == 0] = -100

    encoding["labels"] = labels
    return encoding


# ── 4b. Eval collate ─────────────────────────────────────────────────────────
# For evaluation we keep the raw choices so the MC scoring function can
# build one candidate string per choice and score each independently.

def eval_collate(batch: list[dict]) -> dict:
    """Returns the raw batch dict — MC scorer processes each sample individually."""
    return {
        "images":   [item["image"]   for item in batch],
        "texts":    [item["text"]    for item in batch],
        "choices":  [item["choices"] for item in batch],
        "ids":      [item["id"]      for item in batch],
        "answers":  [item["answer"]  for item in batch],
    }


# ── 4c. DataLoaders ───────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH,
    shuffle=True,
    collate_fn=train_collate,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=EVAL_BATCH,
    shuffle=False,
    collate_fn=eval_collate,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=EVAL_BATCH,
    shuffle=False,
    collate_fn=eval_collate,
    num_workers=2,
    pin_memory=True,
)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

Train batches : 778
Val batches   : 131
Test batches  : 126


## 5. Multiple-Choice Log-Likelihood Scoring

Instead of generating tokens greedily, we score each candidate answer by computing its log-likelihood under the model.  
This is **exact, deterministic, and ~10× faster** than beam-search decoding.

In [ ]:
# ── 5. Batched multiple-choice scoring ───────────────────────────────────────

@torch.inference_mode()
def score_choices_batch(
    model,
    processor,
    images: list[Image.Image],
    prompt_texts: list[str],
    choices_list: list[list[str]],
) -> list[int]:
    """
    For each (image, prompt, choices) triple, score every candidate answer
    by computing the sum of log-probs of the answer token(s) conditioned on
    the prompt.  Returns a list of predicted 0-indexed answer indices.

    Algorithm
    ---------
    For sample i with K_i choices we build K_i full sequences:
        full_i_k = prompt_i + " " + CHOICE_LETTERS[k]
    We tokenise all (sum_i K_i) sequences in one batch call, run a single
    forward pass, and accumulate log-probs of the answer token(s) at their
    positions.  The argmax over K_i gives the predicted answer.

    Parameters
    ----------
    model         : PEFT-wrapped VLM (eval mode)
    processor     : matching AutoProcessor
    images        : list of PIL images  (length = B)
    prompt_texts  : list of prompt strings ending with "Answer:"  (length = B)
    choices_list  : list of choice-string lists  (length = B)

    Returns
    -------
    preds : list[int]  (length = B)
    """
    model.eval()

    all_texts   = []   # full sequences for every (sample, choice)
    all_images  = []   # repeated image per candidate
    sample_idxs = []   # which sample each row belongs to
    choice_idxs = []   # which choice index each row represents

    for s_idx, (prompt, choices, img) in enumerate(zip(prompt_texts, choices_list, images)):
        for c_idx, _ in enumerate(choices):
            answer_letter = CHOICE_LETTERS[c_idx]
            full_seq      = prompt + f" {answer_letter}"
            all_texts.append(full_seq)
            all_images.append(img)
            sample_idxs.append(s_idx)
            choice_idxs.append(c_idx)

    # ── Tokenise all candidates in one batch ──────────────────────────────────
    enc = processor(
        text=all_texts,
        images=all_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    enc = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in enc.items()}

    # ── Also tokenise just the prompts (to find answer-token positions) ───────
    prompt_enc = processor(
        text=[
            prompt_texts[s_idx] for s_idx in sample_idxs
        ],
        images=all_images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    prompt_lens = prompt_enc["attention_mask"].sum(dim=1)  # (N_cands,)

    # ── Forward pass ──────────────────────────────────────────────────────────
    with torch.amp.autocast("cuda", dtype=amp_autocast_dtype, enabled=torch.cuda.is_available()):
        logits = model(**enc).logits   # (N_cands, L, V)

    log_probs = F.log_softmax(logits, dim=-1)   # (N_cands, L, V)

    # ── Accumulate log-prob of answer token(s) ────────────────────────────────
    input_ids  = enc["input_ids"]    # (N_cands, L)
    N          = input_ids.shape[0]

    cand_scores = torch.zeros(N, device=logits.device)
    for n in range(N):
        plen = int(prompt_lens[n].item())
        seq_len = int(enc["attention_mask"][n].sum().item())
        # sum log-probs of tokens from plen-1 → seq_len-1
        # (shifted by 1: logits[t] predicts token[t+1])
        for t in range(plen - 1, seq_len - 1):
            next_token = input_ids[n, t + 1]
            cand_scores[n] += log_probs[n, t, next_token]

    # ── Argmax per sample ─────────────────────────────────────────────────────
    B     = len(prompt_texts)
    preds = []
    for s_idx in range(B):
        mask = [i for i, si in enumerate(sample_idxs) if si == s_idx]
        best = int(torch.argmax(cand_scores[mask]).item())
        preds.append(best)

    return preds


print("Multiple-choice scoring function defined.")

Multiple-choice scoring function defined.


## 6. Training Loop

In [ ]:
# ── 6a. Optimizer & Scheduler ─────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)

total_steps   = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total optimizer steps : {total_steps:,}")
print(f"Warmup steps          : {warmup_steps:,}")

Total optimizer steps : 975
Warmup steps          : 48


In [ ]:
# ── 6b. Helper: validate on the entire val set ────────────────────────────────
def run_validation(model, val_loader) -> tuple[float, float]:
    """
    Returns (val_loss, val_accuracy).
    val_loss     — average NLL of the correct answer token over the val set.
    val_accuracy — fraction of questions answered correctly via MC scoring.
    """
    model.eval()
    total_loss  = 0.0
    total_steps = 0
    correct      = 0
    total       = 0

    for batch in val_loader:
        images   = batch["images"]
        texts    = batch["texts"]
        choices  = batch["choices"]
        answers  = batch["answers"]

        # ── MC accuracy ──────────────────────────────────────────────────────
        preds = score_choices_batch(model, processor, images, texts, choices)
        for pred, gt in zip(preds, answers):
            correct += int(pred == gt)
            total   += 1

        # ── Val loss (NLL on correct answer) ─────────────────────────────────
        # Build full sequences with the ground-truth answer appended
        gt_texts = [
            texts[i] + f" {CHOICE_LETTERS[answers[i]]}"
            for i in range(len(texts))
        ]
        enc = processor(
            text=gt_texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        )
        enc = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in enc.items()}

        prompt_enc = processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        )
        prompt_lens = prompt_enc["attention_mask"].sum(dim=1)

        labels = enc["input_ids"].clone()
        for i, plen in enumerate(prompt_lens):
            labels[i, :plen] = -100
        labels[enc["attention_mask"] == 0] = -100
        enc["labels"] = labels

        with torch.inference_mode():
            with torch.amp.autocast("cuda", dtype=amp_autocast_dtype, enabled=torch.cuda.is_available()):
                loss = model(**enc).loss
        total_loss  += loss.item()
        total_steps += 1

    val_loss = total_loss / max(total_steps, 1)
    val_acc  = correct / max(total, 1)
    return val_loss, val_acc

In [ ]:
# ── 6c. Main training loop ────────────────────────────────────────────────────
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available() and amp_autocast_dtype == torch.float16)

train_metrics = []   # will be saved to disk at end
best_val_acc  = -1.0
global_step   = 0
train_start   = time.time()   # ← total training timer

EARLY_STOP_PATIENCE  = 2
epochs_no_improve    = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss     = 0.0
    epoch_steps    = 0
    optimizer.zero_grad()

    for batch_idx, batch_i in enumerate(train_loader):
        # Move tensors to device
        batch = {k: v.to(model.device, non_blocking=True) if torch.is_tensor(v) else v
                 for k, v in batch_i.items()}
        # Explicitly cast attention_mask to bool and ensure contiguity
        # if "attention_mask" in batch and batch["attention_mask"] is not None:
        #     batch["attention_mask"] = batch["attention_mask"].bool().contiguous()

        with torch.amp.autocast("cuda", dtype=amp_autocast_dtype, enabled=torch.cuda.is_available()):
        # with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss    = outputs.loss / GRAD_ACCUM

        # Only scale if float16 is used for autocasting
        if amp_autocast_dtype == torch.float16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(train_loader):
            if amp_autocast_dtype == torch.float16:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), 1.0
            )
            if amp_autocast_dtype == torch.float16:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        epoch_loss  += loss.item() * GRAD_ACCUM   # unscale for logging
        epoch_steps += 1

        if (batch_idx + 1) % 100 == 0:
            avg = epoch_loss / epoch_steps
            lr_now = scheduler.get_last_lr()[0]
            print(f"  Epoch {epoch} | step {batch_idx+1}/{len(train_loader)} "
                  f"| loss {avg:.4f} | lr {lr_now:.2e}")

    avg_train_loss = epoch_loss / epoch_steps

    # ── End-of-epoch validation ───────────────────────────────────────────────
    print(f"\nEpoch {epoch} complete — running validation…")
    val_loss, val_acc = run_validation(model, val_loader)

    epoch_record = {
        "epoch":          epoch,
        "train_loss":     round(avg_train_loss, 6),
        "val_loss":       round(val_loss,       6),
        "val_accuracy":   round(val_acc,        6),
        "global_step":    global_step,
        "lr":             scheduler.get_last_lr()[0],
        "epoch_time_sec": round(time.time() - train_start, 1),
    }
    train_metrics.append(epoch_record)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch:2d} | train_loss={avg_train_loss:.4f} "
          f"| val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")
    print(f"{'='*60}\n")

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0   # IMPROVEMENT 8: reset patience counter
        model.save_pretrained("best_checkpoint")
        processor.save_pretrained("best_checkpoint")
        print(f"  ✓ New best model saved (val_acc={best_val_acc:.4f})")
    else:
        epochs_no_improve += 1  # IMPROVEMENT 8: increment patience counter
        print(f"  No improvement ({epochs_no_improve}/{EARLY_STOP_PATIENCE} patience epochs used)")

    # IMPROVEMENT 8: halt if patience exhausted
    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch} epochs (no val_acc improvement "
              f"for {EARLY_STOP_PATIENCE} consecutive epochs).")
        break

# ── Save training metrics to disk ─────────────────────────────────────────────
total_training_time = round(time.time() - train_start, 1)
val_wrong_answers   = round((1 - best_val_acc) * len(val_df))
with open(DRIVE_DIR_PATH + f"/train_metrics{SUB_RUN_ID}.json", "w") as f:
    json.dump(
        {
            "config":               cfg,
            "best_val_acc":         best_val_acc,
            "total_training_time_sec": total_training_time,
            "val_wrong_answers":    val_wrong_answers,
            "epochs":               train_metrics,
        },
        f,
        indent=2,
    )

print(f"\nTraining complete in {total_training_time:.1f}s — metrics saved to train_metrics.json")


Epoch 1 complete — running validation…

Epoch  1 | train_loss=1.1685 | val_loss=0.9988 | val_acc=0.5048

  ✓ New best model saved (val_acc=0.5048)

Epoch 2 complete — running validation…

Epoch  2 | train_loss=0.6367 | val_loss=1.0350 | val_acc=0.5810

  ✓ New best model saved (val_acc=0.5810)

Epoch 3 complete — running validation…

Epoch  3 | train_loss=0.4227 | val_loss=1.0900 | val_acc=0.6095

  ✓ New best model saved (val_acc=0.6095)

Epoch 4 complete — running validation…

Epoch  4 | train_loss=0.2773 | val_loss=1.2045 | val_acc=0.6000

  No improvement (1/2 patience epochs used)

Epoch 5 complete — running validation…

Epoch  5 | train_loss=0.2074 | val_loss=1.2634 | val_acc=0.6000

  No improvement (2/2 patience epochs used)

Early stopping triggered after 5 epochs (no val_acc improvement for 2 consecutive epochs).

Training complete in 1537.0s — metrics saved to train_metrics.json


In [ ]:
# ── 6d. Save best checkpoint to DRIVE ────────────────────────────────────────────────────

import shutil

# Define the target path in Google Drive
drive_checkpoint_path = DRIVE_DIR_PATH + "best_checkpoint"
# Copy the local checkpoint to Google Drive
print(f"Saving best_checkpoint to {drive_checkpoint_path}...")
shutil.copytree("best_checkpoint", drive_checkpoint_path)
print("best_checkpoint model and processor saved to Google Drive.")

Saving best_checkpoint to /content/drive/MyDrive/Colab Notebooks/run_3/best_checkpoint...
best_checkpoint model and processor saved to Google Drive.


In [ ]:
# ── 6 FINAL. Run garbage collection ────────────────────────────────────────────────────
import gc

gc.collect()
torch.cuda.empty_cache()
print("Garbage collection complete and CUDA cache emptied.")

Garbage collection complete and CUDA cache emptied.


## 7. Inference Example

Loads the best checkpoint and runs it on one validation sample — matches the starter notebook style.

In [ ]:
import shutil
import os

# Assuming RUN_ID is already defined in the notebook context
# If not, you might need to re-evaluate the cell that defines RUN_ID
CUSTOM_RUN_ID = "run_5" # Re-define RUN_ID for this isolated cell if necessary
MODEL_DIR_PATH = '/content/drive/MyDrive/Colab Notebooks/' + CUSTOM_RUN_ID + '/'

source_path = MODEL_DIR_PATH + "best_checkpoint"
destination_path = "./best_checkpoint"

# Ensure the destination directory exists and is empty before copying
if os.path.exists(destination_path):
    shutil.rmtree(destination_path)

print(f"Copying best_checkpoint from {source_path} to {destination_path}...")
shutil.copytree(source_path, destination_path)
print("best_checkpoint copied successfully.")

Copying best_checkpoint from /content/drive/MyDrive/Colab Notebooks/run_5/best_checkpoint to ./best_checkpoint...
best_checkpoint copied successfully.


In [ ]:
# ── 7. Inference example on a single val sample ───────────────────────────────
from peft import PeftModel

# Reload base + LoRA adapter from best checkpoint
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if use_bnb else None,
    dtype=model_load_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
if not torch.cuda.is_available():
    base_model = base_model.to(device)

inf_model = PeftModel.from_pretrained(base_model, "best_checkpoint")
inf_model.eval()

inf_processor = AutoProcessor.from_pretrained("best_checkpoint")
if inf_processor.tokenizer.pad_token is None:
    inf_processor.tokenizer.pad_token = inf_processor.tokenizer.eos_token

# Pick one sample
sample      = val_df.iloc[0]
sample_img  = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

preds = score_choices_batch(
    inf_model,
    inf_processor,
    images        = [sample_img],
    prompt_texts  = [sample_prompt],
    choices_list  = [sample["choices"]],
)

pred_idx    = preds[0]
pred_letter = CHOICE_LETTERS[pred_idx]
gt_idx      = int(sample["answer"])
gt_letter   = CHOICE_LETTERS[gt_idx]

print("=== PROMPT ===")
print(sample_prompt)
print()
print(f"Predicted answer : {pred_letter} ({sample['choices'][pred_idx]})")
print(f"Ground-truth     : {gt_letter}  ({sample['choices'][gt_idx]})")
print(f"Correct          : {pred_idx == gt_idx}")

=== PROMPT ===
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't al

## 7b. Ablation: Context Fields (hint / lecture)

Each prompt can include up to two context fields from the dataset:
- **`lecture`** — background knowledge relevant to the question topic
- **`hint`** — a passage directly relevant to answering the specific question

This ablation runs log-likelihood scoring on 200 validation samples under
four conditions, holding the model fixed:

| Condition | lecture | hint |
|---|:---:|:---:|
| **Full** (baseline) | ✓ | ✓ |
| **Lecture only** | ✓ | ✗ |
| **Hint only** | ✗ | ✓ |
| **No context** | ✗ | ✗ |

Reported metrics: accuracy per condition, delta vs full, and per-subject
breakdown (natural science / social science). Results saved to
`context_ablation.json`.

In [ ]:
# ── 7b. Ablation: Context Fields (hint / lecture) ─────────────────────────────
# Requires: inf_model and inf_processor loaded in cell 7.
# Scores 200 val samples under 4 context conditions and reports accuracy
# deltas. The model weights are unchanged across all conditions — only the
# prompt text varies, so any accuracy change is purely attributable to the
# presence or absence of that context field.

import time

N_ABLATE = 2000   # number of val samples; reduce if time-constrained

ablate_df = val_df.sample(
    n=min(N_ABLATE, len(val_df)), random_state=SEED
).reset_index(drop=True)
print(f"Ablating on {len(ablate_df)} validation samples")

# ── Coverage: how many samples actually have each field ───────────────────────
# Ablating a field only matters for samples that have it; samples where the
# field is already empty are unaffected by toggling it off.
has_lecture = ablate_df["lecture"].apply(
    lambda v: pd.notna(v) and str(v).strip() != ""
)
has_hint = ablate_df["hint"].apply(
    lambda v: pd.notna(v) and str(v).strip() != ""
)
print(f"  Samples with lecture : {has_lecture.sum()} / {len(ablate_df)}"
      f" ({100*has_lecture.mean():.1f}%)")
print(f"  Samples with hint    : {has_hint.sum()} / {len(ablate_df)}"
      f" ({100*has_hint.mean():.1f}%)\n")


# ── Ablated prompt builder ────────────────────────────────────────────────────
# A thin wrapper around the existing build_prompt logic with use_lecture and
# use_hint flags. Defined locally so it does not touch the original build_prompt
# used everywhere else in the notebook.
def build_prompt_ablated(
    row: pd.Series,
    use_lecture: bool = True,
    use_hint: bool = True,
    use_subject: bool = False,
    use_topic: bool = False,
    use_category: bool = False,
    use_skill: bool = False,
    use_task: bool = False,
    include_answer: bool = False,
) -> str:
    """build_prompt with per-field ablation flags."""
    context_parts = []
    if use_lecture:
        lecture = row.get("lecture", "")
        if pd.notna(lecture) and str(lecture).strip():
            context_parts.append(str(lecture).strip())
    if use_hint:
        hint = row.get("hint", "")
        if pd.notna(hint) and str(hint).strip():
            context_parts.append(str(hint).strip())

    if use_subject:
        subject = row.get("subject", "")
        if pd.notna(subject) and str(subject).strip():
            context_parts.append(str(subject).strip())

    if use_topic:
        topic = row.get("topic", "")
        if pd.notna(topic) and str(topic).strip():
            context_parts.append(str(topic).strip())

    if use_category:
        cat = row.get("category", "")
        if pd.notna(cat) and str(cat).strip():
            context_parts.append(str(cat).strip())

    if use_skill:
        skill = row.get("skill", "")
        if pd.notna(skill) and str(skill).strip():
            context_parts.append(str(skill).strip())

    if use_task:
        task = row.get("task", "")
        if pd.notna(task) and str(task).strip():
            context_parts.append(str(task).strip())

    context_str = "\n".join(context_parts)
    choices_str  = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(row["choices"])
    )

    prompt  = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"

    return prompt

Ablating on 1048 validation samples
  Samples with lecture : 915 / 1048 (87.3%)
  Samples with hint    : 816 / 1048 (77.9%)



In [ ]:
# ── Four ablation conditions ──────────────────────────────────────────────────
CONDITIONS = [
    {"label": "Full (lecture + hint)",  "use_lecture": True,  "use_hint": True},
    {"label": "Lecture only",           "use_lecture": True,  "use_hint": False},
    {"label": "Hint only",              "use_lecture": False, "use_hint": True},
    {"label": "No context",             "use_lecture": False, "use_hint": False},
]

ground_truth = ablate_df["answer"].tolist()
subjects     = ablate_df["subject"].tolist()   # for per-subject breakdown
all_subjects = sorted(set(subjects))

results = []   # one dict per condition

for cond in CONDITIONS:
    label       = cond["label"]
    use_lecture = cond["use_lecture"]
    use_hint    = cond["use_hint"]

    print(f"Running: {label}...")
    t0 = time.time()

    # Build prompts for this condition
    imgs    = [
        Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        for _, row in ablate_df.iterrows()
    ]
    prompts = [
        build_prompt_ablated(row, use_lecture=use_lecture, use_hint=use_hint)
        for _, row in ablate_df.iterrows()
    ]
    choices = [row["choices"] for _, row in ablate_df.iterrows()]

    # Score in mini-batches
    preds = []
    for start in range(0, len(ablate_df), EVAL_BATCH):
        batch_preds = score_choices_batch(
            inf_model, inf_processor,
            imgs[start : start + EVAL_BATCH],
            prompts[start : start + EVAL_BATCH],
            choices[start : start + EVAL_BATCH],
        )
        preds.extend(batch_preds)

    elapsed = time.time() - t0

    # Overall accuracy
    correct = [int(p == g) for p, g in zip(preds, ground_truth)]
    acc     = sum(correct) / len(correct)

    # Per-subject accuracy
    subj_acc = {}
    for subj in all_subjects:
        idxs = [i for i, s in enumerate(subjects) if s == subj]
        subj_acc[subj] = sum(correct[i] for i in idxs) / len(idxs)

    # Accuracy on samples where each field was actually present
    # (ablating an absent field has no effect; these numbers isolate the
    # impact on samples where the field was genuinely removed)
    acc_on_has_lecture = (
        sum(correct[i] for i in has_lecture[has_lecture].index
            if i < len(correct)) /
        max(has_lecture.sum(), 1)
    )
    acc_on_has_hint = (
        sum(correct[i] for i in has_hint[has_hint].index
            if i < len(correct)) /
        max(has_hint.sum(), 1)
    )

    results.append({
        "label":             label,
        "use_lecture":       use_lecture,
        "use_hint":          use_hint,
        "accuracy":          round(acc, 6),
        "correct":           sum(correct),
        "n":                 len(correct),
        "time_sec":          round(elapsed, 2),
        "per_subject":       {k: round(v, 6) for k, v in subj_acc.items()},
        "acc_on_has_lecture": round(acc_on_has_lecture, 6),
        "acc_on_has_hint":    round(acc_on_has_hint, 6),
        "preds":             preds,    # kept for pairwise analysis below
    })
    print(f"  acc={acc:.4f}  ({sum(correct)}/{len(correct)})  "
          f"time={elapsed:.1f}s")


# ── Compute deltas vs Full condition ─────────────────────────────────────────
baseline_acc   = results[0]["accuracy"]
baseline_preds = results[0]["preds"]

for r in results:
    r["delta"]         = round(r["accuracy"] - baseline_acc, 6)
    # Agreement: fraction of samples where this condition agrees with Full
    r["agreement_with_full"] = round(
        sum(p == q for p, q in zip(r["preds"], baseline_preds)) / len(baseline_preds), 6
    )


# ── Print results table ───────────────────────────────────────────────────────
print()
print("=" * 74)
print(f"{'Condition':<26} {'Acc':>7} {'Delta':>8} {'Agree w/ Full':>14}"
      + "".join(f" {s[:10]:>12}" for s in all_subjects))
print("=" * 74)
for r in results:
    delta_str = f"{r['delta']:+.4f}" if r['delta'] != 0 else "baseline"
    subj_str  = "".join(f" {r['per_subject'].get(s, 0):>12.4f}" for s in all_subjects)
    print(f"{r['label']:<26} {r['accuracy']:>7.4f} {delta_str:>8} "
          f"{r['agreement_with_full']:>14.4f}{subj_str}")
print("=" * 74)

# Separate table: accuracy only on samples that actually had each field
print()
print("Accuracy restricted to samples that HAD the ablated field:")
print(f"  {'Condition':<26} {'on has_lecture':>15} {'on has_hint':>12}")
print("  " + "-" * 55)
for r in results:
    print(f"  {r['label']:<26} {r['acc_on_has_lecture']:>15.4f} "
          f"{r['acc_on_has_hint']:>12.4f}")


# ── Save to disk ──────────────────────────────────────────────────────────────
# Strip preds lists before saving (large and not useful in the JSON)
save_results = [{k: v for k, v in r.items() if k != "preds"} for r in results]
out_path = DRIVE_DIR_PATH + f"context_ablation_{SUB_RUN_ID}.json"
with open(out_path, "w") as f:
    json.dump(
        {
            "n_samples":      len(ablate_df),
            "coverage": {
                "has_lecture": int(has_lecture.sum()),
                "has_hint":    int(has_hint.sum()),
            },
            "conditions":     save_results,
        },
        f, indent=2,
    )
print(f"\nResults saved → {out_path}")


Ablating on 1048 validation samples
  Samples with lecture : 915 / 1048 (87.3%)
  Samples with hint    : 816 / 1048 (77.9%)

Running: Full (lecture + hint)...
  acc=0.7872  (825/1048)  time=103.4s
Running: Lecture only...
  acc=0.6842  (717/1048)  time=98.9s
Running: Hint only...
  acc=0.6756  (708/1048)  time=96.8s
Running: No context...
  acc=0.6097  (639/1048)  time=96.3s

Condition                      Acc    Delta  Agree w/ Full   language s   natural sc   social sci
Full (lecture + hint)       0.7872 baseline         1.0000       0.6071       0.7658       0.8765
Lecture only                0.6842  -0.1031         0.8426       0.5714       0.6384       0.8436
Hint only                   0.6756  -0.1116         0.7767       0.4286       0.6422       0.8107
No context                  0.6097  -0.1775         0.6679       0.4286       0.5637       0.7778

Accuracy restricted to samples that HAD the ablated field:
  Condition                   on has_lecture  on has_hint
  -----------

In [ ]:
# ── Four ablation conditions ──────────────────────────────────────────────────
CONDITIONS = [
    {"label": "Full (lecture + hint)",  "use_lecture": True,  "use_hint": True},
    {"label": "Full (all)",  "use_lecture": True,  "use_hint": True, "use_task": True, "use_subject": True, "use_topic": True, "use_category": True, "use_skill": True},
    {"label": "task only",          "use_lecture": True,  "use_hint": True, "use_task": True},
    {"label": "subject only",              "use_lecture": True, "use_hint": True, "use_subject": True},
    {"label": "topic_only",             "use_lecture": True, "use_hint": True, "use_topic": True},
    {"label": "category_only",          "use_lecture": True, "use_hint": True, "use_category": True},
    {"label": "skill_only",             "use_lecture": True, "use_hint": True, "use_skill": True},
]

ground_truth = ablate_df["answer"].tolist()
subjects     = ablate_df["subject"].tolist()   # for per-subject breakdown
all_subjects = sorted(set(subjects))

results = []   # one dict per condition

for cond in CONDITIONS:
    label       = cond["label"]
    use_lecture = cond["use_lecture"]
    use_hint    = cond["use_hint"]
    use_task    = cond.get("use_task", False)
    use_subject = cond.get("use_subject", False)
    use_topic   = cond.get("use_topic", False)
    use_category= cond.get("use_category", False)
    use_skill   = cond.get("use_skill", False)

    print(f"Running: {label}...")
    t0 = time.time()

    # Build prompts for this condition
    imgs    = [
        Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        for _, row in ablate_df.iterrows()
    ]
    prompts = [
        build_prompt_ablated(row, use_lecture=use_lecture,
                             use_hint=use_hint,
                             use_task=use_task,
                             use_subject=use_subject,
                             use_topic=use_topic,
                             use_category=use_category,
                             use_skill=use_skill
                             )
        for _, row in ablate_df.iterrows()
    ]
    choices = [row["choices"] for _, row in ablate_df.iterrows()]

    # Score in mini-batches
    preds = []
    for start in range(0, len(ablate_df), EVAL_BATCH):
        batch_preds = score_choices_batch(
            inf_model, inf_processor,
            imgs[start : start + EVAL_BATCH],
            prompts[start : start + EVAL_BATCH],
            choices[start : start + EVAL_BATCH],
        )
        preds.extend(batch_preds)

    elapsed = time.time() - t0

    # Overall accuracy
    correct = [int(p == g) for p, g in zip(preds, ground_truth)]
    acc     = sum(correct) / len(correct)

    # Per-subject accuracy
    subj_acc = {}
    for subj in all_subjects:
        idxs = [i for i, s in enumerate(subjects) if s == subj]
        subj_acc[subj] = sum(correct[i] for i in idxs) / len(idxs)

    # Accuracy on samples where each field was actually present
    # (ablating an absent field has no effect; these numbers isolate the
    # impact on samples where the field was genuinely removed)
    acc_on_has_lecture = (
        sum(correct[i] for i in has_lecture[has_lecture].index
            if i < len(correct)) /
        max(has_lecture.sum(), 1)
    )
    acc_on_has_hint = (
        sum(correct[i] for i in has_hint[has_hint].index
            if i < len(correct)) /
        max(has_hint.sum(), 1)
    )

    results.append({
        "label":             label,
        "use_lecture":       use_lecture,
        "use_hint":          use_hint,
        "use_task":          use_task,
        "use_subject":       use_subject,
        "use_topic":         use_topic,
        "use_category":      use_category,
        "use_skill":         use_skill,
        "accuracy":          round(acc, 6),
        "correct":           sum(correct),
        "correct_df":        correct,
        "n":                 len(correct),
        "time_sec":          round(elapsed, 2),
        "per_subject":       {k: round(v, 6) for k, v in subj_acc.items()},
        "acc_on_has_lecture": round(acc_on_has_lecture, 6),
        "acc_on_has_hint":    round(acc_on_has_hint, 6),
        "preds":             preds,    # kept for pairwise analysis below
    })
    print(f"  acc={acc:.4f}  ({sum(correct)}/{len(correct)})  "
          f"time={elapsed:.1f}s")


# ── Compute deltas vs Full condition ─────────────────────────────────────────
baseline_acc   = results[0]["accuracy"]
baseline_preds = results[0]["preds"]

for r in results:
    r["delta"]         = round(r["accuracy"] - baseline_acc, 6)
    # Agreement: fraction of samples where this condition agrees with Full
    r["agreement_with_full"] = round(
        sum(p == q for p, q in zip(r["preds"], baseline_preds)) / len(baseline_preds), 6
    )


# ── Print results table ───────────────────────────────────────────────────────
print()
print("=" * 74)
print(f"{'Condition':<26} {'Acc':>7} {'Delta':>8} {'Agree w/ Full':>14}"
      + "".join(f" {s[:10]:>12}" for s in all_subjects))
print("=" * 74)
for r in results:
    delta_str = f"{r['delta']:+.4f}" if r['delta'] != 0 else "baseline"
    subj_str  = "".join(f" {r['per_subject'].get(s, 0):>12.4f}" for s in all_subjects)
    print(f"{r['label']:<26} {r['accuracy']:>7.4f} {delta_str:>8} "
          f"{r['agreement_with_full']:>14.4f}{subj_str}")
print("=" * 74)

Running: Full (lecture + hint)...
  acc=0.7872  (825/1048)  time=102.0s
Running: Full (all)...
  acc=0.7719  (809/1048)  time=102.3s
Running: task only...
  acc=0.7805  (818/1048)  time=101.6s
Running: subject only...
  acc=0.7796  (817/1048)  time=103.5s
Running: topic_only...
  acc=0.7786  (816/1048)  time=102.9s
Running: category_only...
  acc=0.7739  (811/1048)  time=103.3s
Running: skill_only...
  acc=0.7901  (828/1048)  time=103.7s

Condition                      Acc    Delta  Agree w/ Full   language s   natural sc   social sci
Full (lecture + hint)       0.7872 baseline         1.0000       0.6071       0.7658       0.8765
Full (all)                  0.7719  -0.0153         0.9189       0.5714       0.7465       0.8765
task only                   0.7805  -0.0067         0.9523       0.6071       0.7529       0.8889
subject only                0.7796  -0.0076         0.9590       0.5714       0.7568       0.8765
topic_only                  0.7786  -0.0086         0.9599       0.

In [ ]:
# Per-category accuracy
def category_acc(cat):
    val_to_subject_list = {val: sub for sub, val in zip(ablate_df["subject"].tolist(), ablate_df[cat].tolist())}
    vals     = ablate_df[cat].tolist()
    all_vals = sorted(set(vals))
    val_acc = {}
    val_baseline_acc = {}
    header = f"Subject    {cat}    "
    for i, r in enumerate(results):
        col = "baseline" if i==0 else r["label"]
        header += f"{col[:10]:>12} "
    print(header)
    print("=" * 74)
    for v in all_vals:
        row = f"{val_to_subject_list[v]:<26} {v:<26} "
        for r in results:
            idxs = [i for i, s in enumerate(vals) if s == v]
            val_acc[v] = sum(r["correct_df"][i] for i in idxs) / len(idxs)
            row += f"{val_acc[v]:>7.4f} "
        print(row)


In [ ]:
category_acc("task")

Subject    task        baseline   Full (all)    task only   subject on   topic_only   category_o   skill_only 
social science             closed choice               0.7921  0.7754  0.7872  0.7852  0.7823  0.7773  0.7931 
natural science            true-or false               0.6364  0.6667  0.5758  0.6061  0.6667  0.6667  0.6970 


In [ ]:
category_acc("subject")

Subject    subject        baseline   Full (all)    task only   subject on   topic_only   category_o   skill_only 
language science           language science            0.6071  0.5714  0.6071  0.5714  0.5714  0.5714  0.5357 
natural science            natural science             0.7658  0.7465  0.7529  0.7568  0.7568  0.7477  0.7658 
social science             social science              0.8765  0.8765  0.8889  0.8765  0.8724  0.8807  0.8971 


In [ ]:
category_acc("topic")

Subject    topic        baseline   Full (all)    task only   subject on   topic_only   category_o   skill_only 
natural science            biology                     0.7912  0.7912  0.8059  0.7949  0.7985  0.7912  0.7985 
natural science            chemistry                   0.6703  0.6044  0.5604  0.5934  0.5714  0.5604  0.6264 
social science             civics                      1.0000  1.0000  1.0000  1.0000  1.0000  1.0000  1.0000 
natural science            earth-science               0.6667  0.6543  0.6543  0.6667  0.6790  0.6667  0.7037 
social science             economics                   1.0000  1.0000  1.0000  1.0000  1.0000  1.0000  1.0000 
social science             geography                   0.8140  0.8062  0.8372  0.8217  0.8062  0.8217  0.8450 
natural science            literacy-in-science         0.4000  0.4000  0.4000  0.4000  0.4000  0.4000  0.4000 
natural science            physics                     0.7136  0.6808  0.7042  0.7136  0.7136  0.6995  0.7136 


In [ ]:
category_acc("category")

Subject    category        baseline   Full (all)    task only   subject on   topic_only   category_o   skill_only 
natural science            Adaptations                 0.9302  0.9302  0.9535  0.9070  0.9302  0.9302  0.9302 
natural science            Adaptations and natural selection  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000 
social science             Age of Exploration          0.0000  1.0000  0.0000  0.0000  0.0000  0.0000  1.0000 
social science             Ancient Egypt and Kush      1.0000  1.0000  1.0000  1.0000  1.0000  1.0000  1.0000 
social science             Ancient Mesopotamia         1.0000  1.0000  1.0000  1.0000  1.0000  1.0000  1.0000 
natural science            Animals                     0.5000  0.5000  0.5000  0.5000  0.5000  0.5000  0.5000 
natural science            Astronomy                   0.6364  0.6667  0.5758  0.6061  0.6667  0.6667  0.6970 
natural science            Atoms and molecules         0.8696  0.8696  0.8696  0.8696  0.8696  0.8696

In [ ]:
category_acc("skill")

Subject    skill        baseline   Full (all)    task only   subject on   topic_only   category_o   skill_only 
natural science            Analyze data to compare properties of planets  0.6364  0.6667  0.5758  0.6061  0.6667  0.6667  0.6970 
social science             Ancient Egyptian religion   1.0000  1.0000  1.0000  1.0000  1.0000  1.0000  1.0000 
natural science            Angiosperm and conifer life cycles  0.5000  0.5000  0.5000  0.5000  0.5000  0.5000  1.0000 
natural science            Animal adaptations: beaks, mouths, and necks  0.9474  0.9474  1.0000  0.8947  0.9474  0.9474  0.9474 
natural science            Animal adaptations: feet and limbs  0.7500  0.7500  0.7500  0.7500  0.7500  0.7500  0.7500 
natural science            Animal adaptations: skins and body coverings  0.9500  0.9500  0.9500  0.9500  0.9500  0.9500  0.9500 
social science             Antebellum Period: slavery and politics part I  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000 
natural science     

In [ ]:
# ── Four ablation conditions ──────────────────────────────────────────────────
CONDITIONS = [
    {"label": "Full (lecture + hint)",  "use_lecture": True,  "use_hint": True,},
    {"label": "Custom (lecture + hint + per_subject)",  "use_lecture": True,  "use_hint": True, "use_custom": True},
]

ground_truth = ablate_df["answer"].tolist()
subjects     = ablate_df["subject"].tolist()   # for per-subject breakdown
all_subjects = sorted(set(subjects))

results = []   # one dict per condition

for cond in CONDITIONS:
    label       = cond["label"]
    use_lecture = cond["use_lecture"]
    use_hint    = cond["use_hint"]
    use_task    = cond.get("use_task", False)
    use_subject = cond.get("use_subject", False)
    use_topic   = cond.get("use_topic", False)
    use_category= cond.get("use_category", False)
    use_skill   = cond.get("use_skill", False)
    use_custom  = cond.get("use_custom", False)

    print(f"Running: {label}...")
    t0 = time.time()

    # Build prompts for this condition
    imgs    = [
        Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        for _, row in ablate_df.iterrows()
    ]
    if not use_custom:
        prompts = [
            build_prompt_ablated(row, use_lecture=use_lecture,
                                use_hint=use_hint,
                                use_task=use_task,
                                use_subject=use_subject,
                                use_topic=use_topic,
                                use_category=use_category,
                                use_skill=use_skill
                                )
            for _, row in ablate_df.iterrows()
        ]
    else:
        prompts = []
        for _, row in ablate_df.iterrows():
            _args = {"use_lecture":use_lecture, "use_hint":use_hint,}

            if row["task"] == "true-or false":
                _args["use_skill"] = True

            if row["subject"] == "language science":
                if row["topic"] == "writing-strategies":
                    _args["use_task"] = True
                if row["category"] == "Persuasive strategies":
                    _args["use_task"] = True

            if row["subject"] == "natural science":
                _args["use_skill"] = True
                if row["topic"] == "biology":
                    _args["use_task"] = True
                if row["topic"] == "chemistry":
                    _args["use_skill"] = False
                if row["topic"] == "earth-science":
                    _args["use_skill"] = True
                if row["category"] == "Classification":
                    _args["use_category"] = True   # try task, subject, topic
                if row["category"] == "Designing experiments":  # baseline
                    _args["use_task"] = False
                    _args["use_subject"] = False
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = False
                if row["category"] == "Ecological interactions" or row["category"] == "Fossils":  # full
                    _args["use_task"] = True
                    _args["use_subject"] = True
                    _args["use_topic"] = True
                    _args["use_category"] = True
                    _args["use_skill"] = True
                if row["category"] == "Ecosystems":
                    _args["use_task"] = True
                    _args["use_skill"] = False   # try reversing this
                if row["category"] == "Magnets":
                    _args["use_task"] = False
                    _args["use_subject"] = True
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = False
                if row["category"] == "Plant reproduction":
                    _args["use_task"] = False
                    _args["use_subject"] = False
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = True
                if row["category"] == "States of matter":
                    _args["use_task"] = False
                    _args["use_subject"] = False
                    _args["use_topic"] = True
                    _args["use_category"] = False
                    _args["use_skill"] = False
                if row["category"] == "Weather and climate":
                    _args["use_task"] = False
                    _args["use_subject"] = False
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = True
                # skill
                if row["category"] == "Compare ages of fossils in a rock sequence":
                    _args["use_task"] = False
                    _args["use_subject"] = True
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = False
                if row["category"] == "Use Punnett squares to calculate probabilities of offspring types":
                    _args["use_task"] = False
                    _args["use_subject"] = False
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = True
                if row["category"] == "Use Punnett squares to calculate ratios of offspring types":
                    _args["use_task"] = False
                    _args["use_subject"] = True
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = False


            if row["subject"] == "social science":
                _args["use_skill"] = True
                if row["topic"] == "geography":
                    _args["use_task"] = True   # try without skill
                if row["topic"] == "world-history":
                    _args["use_skill"] = False  # try skill=True
                if row["category"] == "cities":
                    _args["use_task"] = True
                    _args["use_subject"] = True
                    _args["use_topic"] = True
                    _args["use_category"] = True
                    _args["use_skill"] = True
                if row["skill"] == "Use a letter-number grid":
                    _args["use_task"] = True
                    _args["use_subject"] = False
                    _args["use_topic"] = False
                    _args["use_category"] = False
                    _args["use_skill"] = False

            _prompt = build_prompt_ablated(row, **_args)
            prompts.append(_prompt)
    choices = [row["choices"] for _, row in ablate_df.iterrows()]

    # Score in mini-batches
    preds = []
    for start in range(0, len(ablate_df), EVAL_BATCH):
        batch_preds = score_choices_batch(
            inf_model, inf_processor,
            imgs[start : start + EVAL_BATCH],
            prompts[start : start + EVAL_BATCH],
            choices[start : start + EVAL_BATCH],
        )
        preds.extend(batch_preds)

    elapsed = time.time() - t0

    # Overall accuracy
    correct = [int(p == g) for p, g in zip(preds, ground_truth)]
    acc     = sum(correct) / len(correct)

    # Per-subject accuracy
    subj_acc = {}
    for subj in all_subjects:
        idxs = [i for i, s in enumerate(subjects) if s == subj]
        subj_acc[subj] = sum(correct[i] for i in idxs) / len(idxs)

    # Accuracy on samples where each field was actually present
    # (ablating an absent field has no effect; these numbers isolate the
    # impact on samples where the field was genuinely removed)
    acc_on_has_lecture = (
        sum(correct[i] for i in has_lecture[has_lecture].index
            if i < len(correct)) /
        max(has_lecture.sum(), 1)
    )
    acc_on_has_hint = (
        sum(correct[i] for i in has_hint[has_hint].index
            if i < len(correct)) /
        max(has_hint.sum(), 1)
    )

    results.append({
        "label":             label,
        "use_lecture":       use_lecture,
        "use_hint":          use_hint,
        "use_task":          use_task,
        "use_subject":       use_subject,
        "use_topic":         use_topic,
        "use_category":      use_category,
        "use_skill":         use_skill,
        "accuracy":          round(acc, 6),
        "correct":           sum(correct),
        "correct_df":        correct,
        "n":                 len(correct),
        "time_sec":          round(elapsed, 2),
        "per_subject":       {k: round(v, 6) for k, v in subj_acc.items()},
        "acc_on_has_lecture": round(acc_on_has_lecture, 6),
        "acc_on_has_hint":    round(acc_on_has_hint, 6),
        "preds":             preds,    # kept for pairwise analysis below
    })
    print(f"  acc={acc:.4f}  ({sum(correct)}/{len(correct)})  "
          f"time={elapsed:.1f}s")


# ── Compute deltas vs Full condition ─────────────────────────────────────────
baseline_acc   = results[0]["accuracy"]
baseline_preds = results[0]["preds"]

for r in results:
    r["delta"]         = round(r["accuracy"] - baseline_acc, 6)
    # Agreement: fraction of samples where this condition agrees with Full
    r["agreement_with_full"] = round(
        sum(p == q for p, q in zip(r["preds"], baseline_preds)) / len(baseline_preds), 6
    )


# ── Print results table ───────────────────────────────────────────────────────
print()
print("=" * 74)
print(f"{'Condition':<26} {'Acc':>7} {'Delta':>8} {'Agree w/ Full':>14}"
      + "".join(f" {s[:10]:>12}" for s in all_subjects))
print("=" * 74)
for r in results:
    delta_str = f"{r['delta']:+.4f}" if r['delta'] != 0 else "baseline"
    subj_str  = "".join(f" {r['per_subject'].get(s, 0):>12.4f}" for s in all_subjects)
    print(f"{r['label']:<26} {r['accuracy']:>7.4f} {delta_str:>8} "
          f"{r['agreement_with_full']:>14.4f}{subj_str}")
print("=" * 74)

Running: Full (lecture + hint)...


In [ ]:
# Overall accuracy
baseline_correct = [int(p == g) for p, g in zip(baseline_preds, ground_truth)]

In [ ]:
# Per-category accuracy
def category_acc(cat):
    val_to_subject_list = {val: sub for sub, val in zip(ablate_df["subject"].tolist(), ablate_df[cat].tolist())}
    vals     = ablate_df[cat].tolist()
    all_vals = sorted(set(vals))
    val_acc = {}
    val_baseline_acc = {}
    for v in all_vals:
        idxs = [i for i, s in enumerate(vals) if s == v]
        val_acc[v] = sum(correct[i] for i in idxs) / len(idxs)
        val_baseline_acc[v] = sum(baseline_correct[i] for i in idxs) / len(idxs)

    print("Subject    Field    Baseline Acc    Acc")
    print("=" * 74)
    for b, c in zip(val_baseline_acc.items(), val_acc.items()):
        k_b, v_b = b
        k, v = c
        print(f"{val_to_subject_list[k]:<26} {k:<26} {v_b:>7.4f} {v:>7.4f}")

In [ ]:
category_acc("topic")

Subject    Field    Baseline Acc    Acc
natural science            biology                     0.7912  0.7985
natural science            chemistry                   0.6703  0.6264
social science             civics                      1.0000  1.0000
natural science            earth-science               0.6667  0.7037
social science             economics                   1.0000  1.0000
social science             geography                   0.8140  0.8450
natural science            literacy-in-science         0.4000  0.4000
natural science            physics                     0.7136  0.7136
language science           reading-comprehension       0.7143  0.5714
natural science            science-and-engineering-practices  0.9649  0.9561
social science             us-history                  0.9474  0.9474
social science             world-history               0.6364  0.7273
language science           writing-strategies          0.5714  0.6190


In [ ]:
category_acc("task")

Subject    Field    Baseline Acc    Acc
social science             closed choice               0.7921  0.7951
natural science            true-or false               0.6364  0.6970


In [ ]:
category_acc("category")

Subject    Field    Baseline Acc    Acc
natural science            Adaptations                 0.9302  0.9302
natural science            Adaptations and natural selection  0.0000  0.0000
social science             Age of Exploration          0.0000  1.0000
social science             Ancient Egypt and Kush      1.0000  1.0000
social science             Ancient Mesopotamia         1.0000  1.0000
natural science            Animals                     0.5000  0.5000
natural science            Astronomy                   0.6364  0.6970
natural science            Atoms and molecules         0.8696  0.8696
social science             Basic economic principles   1.0000  1.0000
natural science            Chemical reactions          1.0000  1.0000
social science             Cities                      0.2857  0.4286
natural science            Classification              0.8500  0.8500
natural science            Classification and scientific names  1.0000  1.0000
social science             Colonia

In [ ]:
category_acc("skill")

Subject    Field    Baseline Acc    Acc
natural science            Analyze data to compare properties of planets  0.6364  0.6970
social science             Ancient Egyptian religion   1.0000  1.0000
natural science            Angiosperm and conifer life cycles  0.5000  1.0000
natural science            Animal adaptations: beaks, mouths, and necks  0.9474  0.9474
natural science            Animal adaptations: feet and limbs  0.7500  0.7500
natural science            Animal adaptations: skins and body coverings  0.9500  0.9500
social science             Antebellum Period: slavery and politics part I  0.0000  0.0000
natural science            Benefits of group behavior: African wild dogs  0.0000  0.0000
social science             Causes of the Civil War: Missouri Compromise to Bleeding Kansas  1.0000  1.0000
language science           Choose the sensory details that match the picture  0.6667  0.6667
social science             Cities of the Midwest       0.5000  0.5000
social science      

## 8. Build & Save Submission CSV

In [ ]:
# ── 8. Batched inference over the test set → submission CSV ───────────────────

all_ids    = []
all_preds  = []
infer_start = time.time()   # ← inference timer

print(f"Running inference on {len(test_ds):,} test examples…")

for batch_num, batch in enumerate(test_loader, start=1):
    images   = batch["images"]
    texts    = batch["texts"]
    choices  = batch["choices"]
    ids      = batch["ids"]

    preds = score_choices_batch(inf_model, inf_processor, images, texts, choices)

    all_ids.extend(ids)
    all_preds.extend(preds)

    if batch_num % 20 == 0 or batch_num == len(test_loader):
        done = batch_num * EVAL_BATCH
        print(f"  {min(done, len(test_ds)):,} / {len(test_ds):,} processed")

# ── Build DataFrame ───────────────────────────────────────────────────────────
submission = pd.DataFrame({"id": all_ids, "answer": all_preds})

# Verify ids match the sample submission
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert set(submission["id"]) == set(sample_sub["id"]), "ID mismatch!"
submission = submission.set_index("id").loc[sample_sub["id"]].reset_index()  # align order

# ── Save with timestamp ───────────────────────────────────────────────────────
infer_elapsed = round(time.time() - infer_start, 1)
ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"{DRIVE_DIR_PATH}/submission_{ts}.csv"
submission.to_csv(out_path, index=False)

# ── Save submission metrics ───────────────────────────────────────────────────
sub_metrics = {
    "submission_file":          out_path,
    "inference_time_sec":       infer_elapsed,
    "total_training_time_sec":  total_training_time,
    "val_wrong_answers":        val_wrong_answers,
    "val_accuracy":             best_val_acc,
    "n_test_examples":          len(submission),
}
with open(f"{DRIVE_DIR_PATH}/submission_metrics_{ts}_{SUB_RUN_ID}.json", "w") as f:
    json.dump(sub_metrics, f, indent=2)

print(f"\nSubmission saved → {out_path}")
print(f"Inference time   : {infer_elapsed:.1f}s")
print(f"Val wrong answers: {val_wrong_answers} / {len(val_df)}")
print(f"Shape            : {submission.shape}")
print(f"Answer distribution:")
print(submission["answer"].value_counts().sort_index())
submission.head()

Running inference on 1,008 test examples…
  160 / 1,008 processed
  320 / 1,008 processed
  480 / 1,008 processed
  640 / 1,008 processed
  800 / 1,008 processed
  960 / 1,008 processed
  1,008 / 1,008 processed

Submission saved → /content/drive/MyDrive/Colab Notebooks/run_3//submission_20260505_220607.csv
Inference time   : 732.1s
Val wrong answers: 41 / 105
Shape            : (1008, 2)
Answer distribution:
answer
0    408
1    306
2    205
3     89
Name: count, dtype: int64


,id,answer
0,test_01750,1
1,test_00128,1
2,test_02891,1
3,test_02425,0
4,test_00930,0


In [ ]:
# ── 8b. Submission CSV via Direct Generation (alternative to log-likelihood) ──
#
# Instead of scoring each choice by log-likelihood, the model generates tokens
# greedily after "Answer:" and we parse the first valid choice letter from the
# output. This is slower per sample but requires no forward pass per choice,
# making it faster when num_choices is large.
#
# Requires: inf_model and inf_processor from cell 7 (section 7).
# Produces:  submission_gen_{ts}.csv  and  submission_gen_metrics_{ts}.json
#            saved to DRIVE_DIR_PATH alongside the log-likelihood submission.

import re, time

# ── Helper: extract the first valid answer letter from generated text ─────────
def parse_generated_answer(generated_text: str, num_choices: int) -> int:
    """
    Scan the generated text for the first letter in A…(num_choices-th letter)
    and convert it to a 0-indexed integer.
    Falls back to 0 if no valid letter is found.

    Parameters
    ----------
    generated_text : decoded string of newly generated tokens only (prompt stripped)
    num_choices    : number of choices for this question (bounds valid letters)

    Returns
    -------
    0-indexed predicted answer integer
    """
    valid_letters = CHOICE_LETTERS[:num_choices]          # e.g. "AB" or "ABCD"
    match = re.search(rf"[{re.escape(valid_letters)}]", generated_text.strip())
    if match:
        return CHOICE_LETTERS.index(match.group(0))
    return 0   # fallback: predict first choice


# ── Generation settings ───────────────────────────────────────────────────────
# max_new_tokens=10: enough to capture a letter with surrounding whitespace or
# punctuation; small enough to keep inference fast.
GEN_MAX_NEW_TOKENS = 10

gen_ids      = []
gen_preds    = []
gen_invalids = 0      # count of samples where no valid letter was decoded
gen_start    = time.time()

inf_model.eval()
print(f"Running direct-generation inference on {len(test_df):,} test examples…")
print(f"(max_new_tokens={GEN_MAX_NEW_TOKENS}, greedy, no sampling)")

# ── Sample-by-sample loop ─────────────────────────────────────────────────────
# Batched generation with left-padded inputs causes position-embedding
# misalignment in decoder models, so we process one sample at a time.
for sample_idx, (_, row) in enumerate(test_df.iterrows()):
    img = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize(
        (IMG_SIZE, IMG_SIZE)
    )
    prompt = build_prompt(row, include_answer=False)  # ends with "Answer:"

    inputs = inf_processor(
        text=[prompt],
        images=[img],
        return_tensors="pt",
    )
    inputs = {
        k: v.to(inf_model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        generated_ids = inf_model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,                            # greedy — deterministic
            pad_token_id=inf_processor.tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens — strip the prompt from output
    n_prompt_tokens = inputs["input_ids"].shape[1]
    new_ids         = generated_ids[0, n_prompt_tokens:]
    new_text        = inf_processor.tokenizer.decode(
        new_ids, skip_special_tokens=True
    )

    pred = parse_generated_answer(new_text, int(row["num_choices"]))
    if not re.search(
        rf"[{re.escape(CHOICE_LETTERS[:int(row['num_choices'])])}]",
        new_text.strip(),
    ):
        gen_invalids += 1

    gen_ids.append(row["id"])
    gen_preds.append(pred)

    if (sample_idx + 1) % 100 == 0 or (sample_idx + 1) == len(test_df):
        elapsed = time.time() - gen_start
        print(
            f"  {sample_idx+1:>5} / {len(test_df):,} "
            f"| {elapsed:.0f}s elapsed "
            f"| {elapsed/(sample_idx+1):.2f}s/sample"
        )

gen_elapsed = round(time.time() - gen_start, 1)

# ── Build & validate DataFrame ────────────────────────────────────────────────
gen_submission = pd.DataFrame({"id": gen_ids, "answer": gen_preds})

sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert set(gen_submission["id"]) == set(sample_sub["id"]), "ID mismatch!"
gen_submission = (
    gen_submission.set_index("id")
    .loc[sample_sub["id"]]
    .reset_index()
)

# ── Save submission CSV ───────────────────────────────────────────────────────
ts_gen      = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
gen_out     = f"{DRIVE_DIR_PATH}/submission_gen_{ts_gen}{SUB_RUN_ID}.csv"
gen_submission.to_csv(gen_out, index=False)

# ── Save metrics ──────────────────────────────────────────────────────────────
gen_metrics = {
    "method":              "direct_generation",
    "submission_file":     gen_out,
    "inference_time_sec":  gen_elapsed,
    "time_per_sample_sec": round(gen_elapsed / len(test_df), 4),
    "max_new_tokens":      GEN_MAX_NEW_TOKENS,
    "n_test_examples":     len(gen_submission),
    "n_invalid_outputs":   gen_invalids,
    "pct_invalid":         round(gen_invalids / len(test_df) * 100, 2),
    "answer_distribution": {
        str(k): int(v)
        for k, v in gen_submission["answer"].value_counts().sort_index().items()
    },
}
gen_metrics_path = (
    f"{DRIVE_DIR_PATH}/submission_gen_metrics_{ts_gen}{SUB_RUN_ID}.json"
)
with open(gen_metrics_path, "w") as f:
    json.dump(gen_metrics, f, indent=2)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nDirect-generation submission saved → {gen_out}")
print(f"Inference time      : {gen_elapsed:.1f}s "
      f"({gen_elapsed/len(test_df):.2f}s/sample)")
print(f"Invalid outputs     : {gen_invalids} / {len(test_df)} "
      f"({gen_invalids/len(test_df)*100:.1f}%) — defaulted to choice 0")
print(f"Answer distribution :")
print(gen_submission["answer"].value_counts().sort_index())
gen_submission.head()


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Running direct-generation inference on 1,008 test examples…
(max_new_tokens=10, greedy, no sampling)
    100 / 1,008 | 179s elapsed | 1.79s/sample
    200 / 1,008 | 336s elapsed | 1.68s/sample
    300 / 1,008 | 482s elapsed | 1.61s/sample
    400 / 1,008 | 628s elapsed | 1.57s/sample
    500 / 1,008 | 774s elapsed | 1.55s/sample
    600 / 1,008 | 920s elapsed | 1.53s/sample
    700 / 1,008 | 1065s elapsed | 1.52s/sample
    800 / 1,008 | 1213s elapsed | 1.52s/sample
    900 / 1,008 | 1358s elapsed | 1.51s/sample
   1000 / 1,008 | 1503s elapsed | 1.50s/sample
   1008 / 1,008 | 1515s elapsed | 1.50s/sample

Direct-generation submission saved → /content/drive/MyDrive/Colab Notebooks/run_3//submission_gen_20260505_232739.csv
Inference time      : 1514.9s (1.50s/sample)
Invalid outputs     : 0 / 1008 (0.0%) — defaulted to choice 0
Answer distribution :
answer
0    407
1    305
2    207
3     89
Name: count, dtype: int64


,id,answer
0,test_01750,1
1,test_00128,1
2,test_02891,1
3,test_02425,0
4,test_00930,0


## 📓 Experiment Diary — Run End

> **Post-run notes:**

In [ ]:
# Final compute diary stamp — run this at the END of each session
from datetime import datetime
import json, os
stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
with open("diary/compute_diary_p3.txt", "a") as f:
    f.write(json.dumps({"event": "session_end", "when": stamp}) + "\n")
print("⏱️ Session end logged:", stamp)

⏱️ Session end logged: 2026-05-05 22:06:07
